# Notebook 01: Exploración del Dataset Filtrado

**Entrada:** `data/processed/iedb_sars_flu_filtered.csv` (generado en el Notebook 00)  
**Salida:** `data/processed/protein_labels.csv` (una fila por proteína con su label)

En este notebook exploramos el dataset ya filtrado para entender:
- La distribución de ensayos positivos y negativos
- Qué proteínas están representadas y cuántos epítopos tiene cada una
- El desbalance de clases que tendremos en el modelo
- Si la proteína Spike de SARS-CoV-2 aparece como la más antigénica (validación informal)

## 0. Configuración

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')

## 1. Carga del dataset

In [ ]:
PATH = PROCESSED_DIR / 'iedb_sars_flu_filtered.csv'

df = pd.read_csv(PATH)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
df.head()

## 2. Estructura y calidad de los datos

In [ ]:
print("Tipos de datos:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())
print(f"\nFilas duplicadas: {df.duplicated().sum():,}")

## 3. Distribución de resultados de ensayo

Los valores `Positive`, `Positive-Low`, `Positive-High` y `Positive-Intermediate`
representan todos evidencia de antigenicidad. Los trataremos todos como positivos.

In [ ]:
print("Distribución de qualitative_measurement:")
counts = df['qualitative_measurement'].value_counts(dropna=False)
print(counts)
print(f"\nTotal positivos (todas las variantes): {counts[counts.index.str.startswith('Positive', na=False)].sum():,}")
print(f"Total negativos: {counts.get('Negative', 0):,}")

In [ ]:
df['is_positive'] = df['qualitative_measurement'].str.startswith('Positive').astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Negative (0)', 'Positive (1)']
values = [df['is_positive'].value_counts()[0], df['is_positive'].value_counts()[1]]
bars = ax.bar(labels, values, color=['#d9534f', '#5cb85c'], width=0.5)
ax.set_title('Distribución de ensayos positivos vs negativos', fontsize=13)
ax.set_ylabel('Número de ensayos')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{val:,}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Distribución por tipo de ensayo

In [ ]:
print("Distribución por tipo de ensayo:")
print(df.groupby('assay_type')['is_positive'].value_counts().unstack(fill_value=0))
print(f"\nTasa de positividad tcell: {df[df['assay_type']=='tcell']['is_positive'].mean():.1%}")
print(f"Tasa de positividad bcell: {df[df['assay_type']=='bcell']['is_positive'].mean():.1%}")

## 5. Distribución por organismo

In [ ]:
df['pathogen'] = 'Other'
df.loc[df['source_organism'].str.contains(
    'SARS-CoV-2|Severe acute respiratory syndrome coronavirus 2',
    case=False, na=False), 'pathogen'] = 'SARS-CoV-2'
df.loc[df['source_organism'].str.contains(
    'Influenza A', case=False, na=False), 'pathogen'] = 'Influenza A'

print("Filas por patógeno:")
print(df['pathogen'].value_counts())
print("\nEnsayos positivos por patógeno:")
print(df.groupby('pathogen')['is_positive'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'positivos', 'count': 'total', 'mean': 'tasa'}))

## 6. Proteínas únicas y su antigenicidad

Aquí pasamos de pensar en epítopos a pensar en proteínas, que es la unidad
de análisis de nuestro modelo.

In [ ]:
protein_summary = df.groupby(['source_molecule_iri', 'source_molecule', 'pathogen']).agg(
    total_epitopes=('epitope_name', 'count'),
    positive_epitopes=('is_positive', 'sum'),
    negative_epitopes=('is_positive', lambda x: (x == 0).sum())
).reset_index()

protein_summary['label'] = (protein_summary['positive_epitopes'] > 0).astype(int)

print(f"Total proteínas únicas: {len(protein_summary):,}")
print(f"\nDistribución de labels a nivel de proteína:")
print(protein_summary['label'].value_counts())
print(f"\nPor patógeno:")
print(protein_summary.groupby(['pathogen', 'label']).size().unstack(fill_value=0))

## 7. Proteínas más representadas por patógeno

Validación informal: la proteína Spike de SARS-CoV-2 debería ser la más representada.

In [ ]:
print("Top 15 proteínas de SARS-CoV-2 por número de epítopos:")
sars_proteins = protein_summary[protein_summary['pathogen'] == 'SARS-CoV-2'].sort_values(
    'total_epitopes', ascending=False).head(15)
print(sars_proteins[['source_molecule', 'total_epitopes', 'positive_epitopes', 'label']].to_string(index=False))

In [ ]:
print("Top 15 proteínas de Influenza A por número de epítopos:")
flu_proteins = protein_summary[protein_summary['pathogen'] == 'Influenza A'].sort_values(
    'total_epitopes', ascending=False).head(15)
print(flu_proteins[['source_molecule', 'total_epitopes', 'positive_epitopes', 'label']].to_string(index=False))

In [ ]:
top_sars = protein_summary[
    (protein_summary['pathogen'] == 'SARS-CoV-2') &
    (protein_summary['label'] == 1)
].sort_values('positive_epitopes', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_sars['source_molecule'].str[:45], top_sars['positive_epitopes'],
               color='#5cb85c')
ax.invert_yaxis()
ax.set_xlabel('Número de epítopos positivos')
ax.set_title('Top 10 proteínas antigénicas de SARS-CoV-2', fontsize=13)
for bar, val in zip(bars, top_sars['positive_epitopes']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(int(val)), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 8. Análisis de proteínas con ensayos mixtos

Proteínas que tienen tanto epítopos positivos como negativos.
Según nuestras reglas de etiquetado, estas se clasifican como `label = 1`.

In [ ]:
mixed = protein_summary[
    (protein_summary['positive_epitopes'] > 0) &
    (protein_summary['negative_epitopes'] > 0)
]
print(f"Proteínas con ensayos mixtos (positivos Y negativos): {len(mixed):,}")
print(f"  De SARS-CoV-2: {len(mixed[mixed['pathogen']=='SARS-CoV-2']):,}")
print(f"  De Influenza A: {len(mixed[mixed['pathogen']=='Influenza A']):,}")
print(f"\nEstas proteínas se etiquetan como label=1 (hay evidencia de antigenicidad).")

## 9. Resumen del dataset

In [ ]:
n_total     = len(df)
n_positive  = df['is_positive'].sum()
n_negative  = (df['is_positive'] == 0).sum()
n_proteins  = len(protein_summary)
n_prot_pos  = (protein_summary['label'] == 1).sum()
n_prot_neg  = (protein_summary['label'] == 0).sum()
n_sars_prot = len(protein_summary[protein_summary['pathogen'] == 'SARS-CoV-2'])
n_flu_prot  = len(protein_summary[protein_summary['pathogen'] == 'Influenza A'])

print("=" * 55)
print("RESUMEN DEL DATASET")
print("=" * 55)
print(f"  Total ensayos (filas):          {n_total:>10,}")
print(f"  Ensayos positivos:              {n_positive:>10,} ({n_positive/n_total:.1%})")
print(f"  Ensayos negativos:              {n_negative:>10,} ({n_negative/n_total:.1%})")
print(f"  Proteínas únicas totales:       {n_proteins:>10,}")
print(f"  Proteínas label=1 (antigénica): {n_prot_pos:>10,}")
print(f"  Proteínas label=0 (negativa):   {n_prot_neg:>10,}")
print(f"  Proteínas SARS-CoV-2:           {n_sars_prot:>10,}")
print(f"  Proteínas Influenza A:          {n_flu_prot:>10,}")
print("=" * 55)
print(f"\nDesbalance de clases (label=1/label=0): {n_prot_pos/n_prot_neg:.2f}")
print("→ Si es muy distinto de 1.0, aplicar class_weight='balanced' en el modelo.")

## 10. Guardar protein_labels.csv para el Notebook 02

Este archivo intermedio tiene una fila por proteína con su label ya calculado.

In [ ]:
OUTPUT_PATH = PROCESSED_DIR / 'protein_labels.csv'

protein_summary.to_csv(OUTPUT_PATH, index=False)
print(f"Guardado: {OUTPUT_PATH}")
print(f"Filas: {len(protein_summary):,} | Columnas: {list(protein_summary.columns)}")